# 02 - Fine Tune Transformer for Fact-Checking

This notebook is responsible for performing the fact-checking task on the claims that were extracted and normalized in the previous notebook. It loads the datasets generated previously and fine-tunes a transformer for fact-checking the claims as true or false.

### Imports

In [ ]:
# Native
import os
import json
import shutil
import logging

# Third-party
import torch
import sklearn
import evaluate
import numpy as np
import pandas as pd
from tqdm import tqdm
from emoji import demojize
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
		Trainer,
    EarlyStoppingCallback,
)

### Setup

In [ ]:
# Configure logging (safe for notebook re-runs)
root_logger = logging.getLogger()

if not root_logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
else:
    # Avoid duplicate handlers when re-running notebook cells: just set levels
    root_logger.setLevel(logging.INFO)
    for h in root_logger.handlers:
        h.setLevel(logging.INFO)
    # Optionally disable propagation to avoid duplicate output from external loggers
    root_logger.propagate = False

### Constants

In [ ]:
# Execution Constants
TIMESTAMP = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M-%S")

# Dataset Constants
DATASET_NAME = "fakebr" # ["faketweetbr", "fakebr"]
DATASET_TASK = "original"  # ["original", "claim_extraction", "claim_normalization"]
DATASET_PROCESS_ID = ""

# Model Constants
MODEL_NAME = "FacebookAI/xlm-roberta-large"  # ["FacebookAI/xlm-roberta-large", "neuralmind/bert-large-portuguese-cased"]
SAVE_MODEL = True # Whether to save the fine-tuned model or not. This is necessary for loading the best model after fine-tuning. BEWARE: it may consume a lot of disk space!

# Paths Constants
DATA_PATH = f"../data/{DATASET_NAME}/{DATASET_TASK}/{DATASET_PROCESS_ID + "/" if DATASET_PROCESS_ID else ""}" # Last path corresponds to the task that original data (i.e., original, claim_extraction, claim_normalization).
OUTPUT_PATH = f"../data/{DATASET_NAME}/fine-tuning/{MODEL_NAME.split('/')[-1]}/{DATASET_TASK}/{DATASET_PROCESS_ID + "/" if DATASET_PROCESS_ID else ""}{TIMESTAMP}"
MODEL_PATH = f"{OUTPUT_PATH}/model/"
METRICS_PATH = f"{OUTPUT_PATH}/metrics/"
RESULTS_PATH = f"../data/{DATASET_NAME}/classification_results/{MODEL_NAME.split('/')[-1]}/"

### Verify GPU Availability and Info

In [ ]:
# Log GPU info
if torch.cuda.is_available():
    logging.info(
        f"Torch CUDA version: {torch.version.cuda}; GPU: {torch.cuda.get_device_name(0)}"
    )
else:
    logging.info("No GPU found, training on CPU")

### Load and Setup Tokenizer

In [ ]:
# Load and Setup Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, do_lower_case=False, normalization=True
)
tokenizer.demoizer = tokenizer.demojizer = lambda x: demojize(x, language="pt")

# Preprocessing Function
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

### Load Dataset

In [ ]:
# Map Label Function
def map_label(example):
    lab = example["label"]

    if isinstance(lab, str):
        example["label"] = label2id.get(lab, None)  # None -> will be filtered if needed

    return example

# Filter Function
def filter_missing_labels(example):
    return example["label"] is not None

# Define dataset files
train_file = DATA_PATH + 'train.csv'
validation_file = DATA_PATH +  'validation.csv'
test_file = DATA_PATH +  'test.csv'

# Define label mappings
label2id = {"true": 0, "fake": 1}
id2label = {v: k for k, v in label2id.items()}

# Load dataset
dataset = load_dataset('csv', data_files={'train': train_file, 'validation': validation_file, 'test': test_file})

# Rename columns
dataset = dataset.rename_column("classificacao", "label")

# Apply label mapping
dataset = dataset.map(map_label, batched=False)

# Tokenize dataset
remove_cols = [c for c in dataset["train"].column_names if c not in ("custom_id", "text", "label")]
tokenized = dataset.map(preprocess_function, batched=True, remove_columns=remove_cols)

# Filter out examples with missing labels
tokenized = tokenized.filter(filter_missing_labels)

### Load Model

In [ ]:
# Load Model
model = AutoModelForSequenceClassification.from_pretrained(
  MODEL_NAME, 
  problem_type="single_label_classification",
  num_labels=2,
	label2id=label2id,
	id2label=id2label,
).to('cuda' if torch.cuda.is_available() else 'cpu')

# Check if model is using GPU or CPU
logging.info(f"Model device: {next(model.parameters()).device}")

### Define Metrics Computation Function

In [ ]:
# Metrics Computation Function
def compute_metrics(eval_pred):
    """Compute metrics for the evaluation"""
    # Unpack predictions and labels
    preds, labels = eval_pred

    # Get predictions
    predictions = np.argmax(preds, axis=-1)

    # Load metrics
    clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

    # Compute and return metrics
    return clf_metrics.compute(predictions=predictions, references=labels)

### Define Training Arguments

In [ ]:
# Define EarlyStoppingCallback with patience of 2 epochs
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,  # Stop training if no improvement for 2 epochs
)

# Training Arguments
training_args = TrainingArguments(
    output_dir=f"{MODEL_PATH}/checkpoints" if SAVE_MODEL else None,
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    num_train_epochs=10,
    logging_strategy="epoch",
    weight_decay=0.01,
    eval_strategy="epoch",
    do_eval=True,
    save_strategy="epoch" if SAVE_MODEL else "no",
    save_total_limit=3,
    load_best_model_at_end=SAVE_MODEL,
    metric_for_best_model="eval_loss",
    fp16=False, # Enable if using NVIDIA GPUs with Tensor Cores
    bf16=True,  # Enable automatic mixed precision (Ada Lovelace Architecture).
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[early_stopping],
)

### Train Model

In [ ]:
# Train model
trainer.train()

### Save Best Model Metrics and Predictions on Test Set

In [ ]:
# Check if paths exists
os.makedirs(METRICS_PATH, exist_ok=True)
os.makedirs(os.path.join(RESULTS_PATH, "classifications"), exist_ok=True)

# Classify Test Set and Save Results
test_results = trainer.predict(tokenized["test"])

# Prepare results DataFrame
test_preds = np.argmax(test_results.predictions, axis=-1)
test_labels = test_results.label_ids

results_df = pd.DataFrame(
    {
        "custom_id": tokenized["test"]["custom_id"],
        "text": tokenized["test"]["text"],
        "original_label": [id2label[label] for label in test_labels],
        "predicted_label": [id2label[pred] for pred in test_preds],
    }
)

# Save results to CSV
results_csv_path = os.path.join(
    RESULTS_PATH,
    f"classifications/{DATASET_TASK.replace("_", "-")}{("_" + DATASET_PROCESS_ID) if DATASET_PROCESS_ID else ""}_test-set-eval_{TIMESTAMP}.csv",
)
results_df.to_csv(results_csv_path, index=False)
logging.info(f"Saved test set evaluation results to {results_csv_path}.")

# Save Metrics
with open(
		os.path.join(
				METRICS_PATH,
				f"best_model_metrics.json"
		),
		"w",
) as f:
		json.dump(test_results.metrics, f, indent=4)

### Save Model

In [ ]:
if SAVE_MODEL:
  	# Check if path exists
    os.makedirs(f"{MODEL_PATH}/best_model", exist_ok=True)

	  # Save model (Trainer save)
    trainer.save_model(f"{MODEL_PATH}/best_model")
    
    # Save Safetensors-safe Weights
    try:
        # Attempt to save weights using safetensors (no torch>=2.6 requirement to load safetensors files)
        model.save_pretrained(f"{MODEL_PATH}/best_model", safe_serialization=True)
        logging.info(f"Saved model weights using safetensors at {MODEL_PATH}")
    except Exception as e:
        logging.warning(
					  "Could not save using safetensors. If you want safetensors output, install the 'safetensors' package: pip install safetensors."
			  )
        
		# Delete only checkpoints
    checkpoints_path = os.path.join(MODEL_PATH, "checkpoints")
    if os.path.exists(checkpoints_path):
        shutil.rmtree(checkpoints_path)
        logging.info(f"Deleted checkpoints directory at {checkpoints_path}.")
else:
		# Delete model directory if it exists
		if os.path.exists(MODEL_PATH):
				shutil.rmtree(MODEL_PATH)
				logging.info(f"Model saving disabled. Deleted model directory at {MODEL_PATH}.")

### Clean GPU VRAM

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()